# DWBI Week 4 — Dimensional Modeling
**Ali Raza** · `su92-bsdsm-f23-018` · **BSDS-7A**

PakFreight's Week-3 star (1 fact + 5 dims, 34 rows, 578,650 kg) faces the snowflake question: should DimCity be normalized? Seven tasks count the joins, build the snowflake, do the storage arithmetic, register the relationships, settle cross-filter direction — and the totals do not move by a single kilogram.

## T6 — When to snowflake (12 marks)

Six scenarios against the four legitimate cases:
**Case 1** — a genuine outrigger (a dimension in its own right, e.g. DimDate at month grain);
**Case 2** — huge dimension, wide high-cardinality text; **Case 3** — owned elsewhere, different cadence;
**Case 4** — multi-valued (bridge, not snowflake).

| # | Scenario | Verdict | Case | Why |
|---|---|---|---|---|
| 1 | DimCity carries Province and Region across 15 rows | **Flatten** | — | T2 proved it: −18 bytes. The joins cost more than the bytes |
| 2 | DimRoute needs the month its corridor opened | **Snowflake** | 1 | Month-opened is a genuine outrigger — conformed DimDate at month grain |
| 3 | 50M-row customer dim repeats a 400-char segment description | **Snowflake** | 2 | The dictionary cannot hold it — real bytes saved |
| 4 | Sub-category loads nightly; manufacturer detail arrives monthly from another vendor | **Snowflake** | 3 | Owned elsewhere, different cadence — separate table, separate load |
| 5 | One shipment can carry several hazard classes | **Bridge** | 4 | Multi-valued — a bridge table, not a snowflake |
| 6 | The diagram would look tidier if City rolled up into Province | **Flatten** | — | "Tidier diagram" is not a reason |

### The test, in my own words

> Snowflake only when **(1)** the attribute is a true dimension in its own right **AND (2)** the win —
> storage, ownership or grain — outweighs the join cost. **Both must hold.**

In [ ]:
# T6 counts: 2 Flatten, 3 Snowflake, 1 Bridge
v = ["Flatten","Snowflake","Snowflake","Snowflake","Bridge","Flatten"]
print({k: v.count(k) for k in set(v)})
assert v.count("Flatten")==2 and v.count("Snowflake")==3 and v.count("Bridge")==1